# Build Features

Joins all input tables into one clean modeling dataset — one row per player.

### Input tables
| Table | Features |
|---|---|
| `draft_history` | round, overall pick, age at draft, college vs. international |
| `draft_combine_stats` | height, weight, wingspan, vertical, agility, sprint |
| `common_player_info` | position, country |

### Output
`features.csv` — one row per player, ready to join with target.

In [14]:
import sqlite3
import pandas as pd

DB_PATH     = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\nba.sqlite"
OUTPUT_PATH = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\features.csv"

conn = sqlite3.connect(DB_PATH)
print("Connected")

Connected


## Step 1 — Draft History Features

In [15]:
draft = pd.read_sql_query("""
    SELECT
        person_id         AS player_id,
        season            AS draft_year,
        round_number,
        overall_pick,
        organization_type
    FROM draft_history
    WHERE season >= 2000
""", conn)

draft['player_id'] = draft['player_id'].astype('int64')

# Binary: came from college (1) vs international/other (0)
draft['from_college'] = (draft['organization_type'] == 'College/University').astype(int)
draft.drop(columns='organization_type', inplace=True)

print(f"Draft rows: {len(draft)}")
draft.head()

Draft rows: 1425


,player_id,draft_year,round_number,overall_pick,from_college
0,2030,2000,1,1,1
1,2031,2000,1,2,1
2,2032,2000,1,3,0
3,2033,2000,1,4,1
4,2034,2000,1,5,1


## Step 2 — Combine Stats Features

In [16]:
combine = pd.read_sql_query("""
    SELECT
        player_id,
        height_wo_shoes,
        weight,
        wingspan,
        standing_reach,
        standing_vertical_leap,
        max_vertical_leap,
        lane_agility_time,
        three_quarter_sprint
    FROM draft_combine_stats
    WHERE season >= 2000
""", conn)

combine['player_id'] = combine['player_id'].astype('int64')

# Some players attended multiple combines — keep the most recent row
combine = combine.drop_duplicates(subset='player_id', keep='last')

print(f"Combine rows: {len(combine)}")
print("\nMissing %:")
print((combine.isnull().mean() * 100).round(1))

Combine rows: 1591

Missing %:
player_id                  0.0
height_wo_shoes            3.3
weight                     3.4
wingspan                   3.3
standing_reach             3.4
standing_vertical_leap    14.5
max_vertical_leap         14.5
lane_agility_time         15.0
three_quarter_sprint      15.0
dtype: float64


## Step 3 — Player Info Features

In [17]:
info = pd.read_sql_query("""
    SELECT
        person_id AS player_id,
        position,
        country
    FROM common_player_info
    WHERE draft_year >= 2000
""", conn)

info['player_id'] = info['player_id'].astype('int64')

# Simplify position to G / F / C
def simplify_position(pos):
    if pd.isna(pos): return 'Unknown'
    if 'G' in pos: return 'G'
    if 'F' in pos: return 'F'
    if 'C' in pos: return 'C'
    return 'Unknown'

info['position_simple'] = info['position'].apply(simplify_position)
info['is_international'] = (info['country'] != 'USA').astype(int)
info.drop(columns=['position', 'country'], inplace=True)

print(f"Player info rows: {len(info)}")
info['position_simple'].value_counts()

Player info rows: 1876


position_simple
G          888
F          732
C          222
Unknown     34
Name: count, dtype: int64

## Step 4 — Join All Features

In [18]:
features = draft.merge(combine, on='player_id', how='left')
features = features.merge(info,    on='player_id', how='left')

print(f"Total players: {len(features)}")
print(f"Players with combine data: {features['wingspan'].notna().sum()}")
print(f"Players without combine data: {features['wingspan'].isna().sum()}")
features.head()

Total players: 1425
Players with combine data: 907
Players without combine data: 518


,player_id,draft_year,round_number,overall_pick,from_college,height_wo_shoes,weight,wingspan,standing_reach,standing_vertical_leap,max_vertical_leap,lane_agility_time,three_quarter_sprint,position_simple,is_international
0,2030,2000,1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2031,2000,1,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F,0.0
2,2032,2000,1,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2033,2000,1,4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,0.0
4,2034,2000,1,5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,G,0.0


## Step 5 — Handle Missing Combine Data

~40% of drafted players didn't attend the combine. We handle this with two steps:
1. Add a `has_combine` flag so the model can learn that missing combine data is itself informative
2. Impute missing values with the median (position-specific where possible)

In [19]:
combine_cols = [
    'height_wo_shoes', 'weight', 'wingspan', 'standing_reach',
    'standing_vertical_leap', 'max_vertical_leap',
    'lane_agility_time', 'three_quarter_sprint'
]

# Flag: did this player attend the combine?
features['has_combine'] = features['wingspan'].notna().astype(int)

# Replace empty strings with NaN, then cast to numeric
for col in combine_cols:
    features[col] = pd.to_numeric(features[col], errors='coerce')

# Impute missing with global median
for col in combine_cols:
    median_val = features[col].median()
    features[col] = features[col].fillna(median_val)

# One-hot encode position
features = pd.get_dummies(features, columns=['position_simple'], prefix='pos')

print("Missing values remaining:")
print(features.isnull().sum()[features.isnull().sum() > 0])
print(f"\nFinal feature shape: {features.shape}")
features.head()

Missing values remaining:
is_international    528
dtype: int64

Final feature shape: (1425, 19)


,player_id,draft_year,round_number,overall_pick,from_college,height_wo_shoes,weight,wingspan,standing_reach,standing_vertical_leap,max_vertical_leap,lane_agility_time,three_quarter_sprint,is_international,has_combine,pos_C,pos_F,pos_G,pos_Unknown
0,2030,2000,1,1,1,77.75,213.0,82.75,103.75,30.0,35.0,11.26,3.26,NaN,0,False,False,False,False
1,2031,2000,1,2,1,77.75,213.0,82.75,103.75,30.0,35.0,11.26,3.26,0.0,0,False,True,False,False
2,2032,2000,1,3,0,77.75,213.0,82.75,103.75,30.0,35.0,11.26,3.26,NaN,0,False,False,False,False
3,2033,2000,1,4,1,77.75,213.0,82.75,103.75,30.0,35.0,11.26,3.26,0.0,0,False,False,False,True
4,2034,2000,1,5,1,77.75,213.0,82.75,103.75,30.0,35.0,11.26,3.26,0.0,0,False,False,True,False


In [20]:
features.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(features)} rows to {OUTPUT_PATH}")

Saved 1425 rows to E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\features.csv
